# Create narration data from frozen Magi DETR tokens
Run this notebook first. It reads only development labels and existing Magi/OCR caches.
Only direct, confident, unique OCR-to-GT matches are accepted; joined bubbles are excluded.
All Magi weights stay frozen. OCR and GT are audit metadata, never classifier inputs.

In [1]:
import hashlib
import json
import random
import re
import unicodedata
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from scipy.optimize import linear_sum_assignment
from transformers import AutoModel
from transformers.image_transforms import center_to_corners_format
from tqdm.auto import tqdm

ROOT = Path(r'C:\Users\ESHAAN\BYTE\Applied-Task\Magi\isNarratorTextMagi')
DEVELOPMENT = ROOT.parent / 'TaskData' / 'development'
CACHE = ROOT.parent / 'paddleOCR-Magi' / 'cache_baseline' / 'development'
MAGI_PATH = r'C:\MODELS\magiv2'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
MATCH_THRESHOLD = 95.0
MATCH_MARGIN = 5.0
MIN_BBOX_IOU = 0.7
TEXT_DETECTION_THRESHOLD = 0.3
VALIDATION_FRACTION = 0.20
NEGATIVE_RATIO = 2.0
ROOT.mkdir(exist_ok=True)
print('Device:', DEVICE)


Device: cuda


## Accept clean OCR-to-GT labels

In [2]:
def normalize_text(text):
    text = unicodedata.normalize('NFKC', text or '').casefold()
    words = ' '.join(''.join(c if c.isalnum() else ' ' for c in text).split())
    if words:
        return words
    symbols = ''.join(c for c in text if c.isspace() or unicodedata.category(c)[0] in {'P', 'S'})
    return ' '.join(symbols.split())


def confident_match(text, gt_lines):
    text = normalize_text(text)
    if not text:
        return None
    scores = []
    for index, gt in enumerate(gt_lines):
        target = normalize_text(gt['text'])
        if target:
            score = 100 * SequenceMatcher(None, text, target, autojunk=False).ratio()
            scores.append((score, index))
    scores.sort(reverse=True)
    if not scores or scores[0][0] < MATCH_THRESHOLD:
        return None
    if len(scores) > 1 and scores[0][0] - scores[1][0] < MATCH_MARGIN:
        return None
    score, index = scores[0]
    return {'gt_index': index, 'match_score': score, 'match_type': 'direct'}


def digest(value):
    text = json.dumps(value, sort_keys=True, ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def check_boxes(boxes, width, height, allow_outside=False):
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    assert np.isfinite(boxes).all()
    assert (boxes[:, 2:] > boxes[:, :2]).all(), 'Expected xyxy boxes.'
    if not allow_outside:
        assert (boxes >= 0).all()
        assert (boxes <= [width, height, width, height]).all(), 'Expected original-page pixels.'
    return boxes


In [3]:
labels = {}
with (DEVELOPMENT / 'labels.jsonl').open(encoding='utf-8') as file:
    for line in file:
        if line.strip():
            row = json.loads(line)
            assert row['sequence_id'] not in labels
            labels[row['sequence_id']] = row

missing = [seq for seq in labels if not (CACHE / seq / 'magi_raw.json').is_file()]
assert not missing, f'Missing development caches: {missing}'
records = []
sequence_pages = {}
excluded = Counter()
for sequence_id in sorted(labels):
    saved = json.loads((CACHE / sequence_id / 'magi_raw.json').read_text(encoding='utf-8'))
    assert saved['settings']['split'] == 'development'
    assert saved['settings']['sequence_id'] == sequence_id
    assert saved['settings']['preprocessing'] == 'grayscale_then_RGB'
    assert digest(saved['data']) == saved['data_hash'], 'Cache checksum mismatch.'
    page_files = sorted((DEVELOPMENT / 'images' / sequence_id).glob('*.png'))
    assert len(page_files) == len(saved['data']) == len(labels[sequence_id]['pages'])
    assert [p.name for p in page_files] == [p['name'] for p in saved['settings']['pages']]
    sequence_pages[sequence_id] = []
    for page_idx, (page_path, page) in enumerate(zip(page_files, saved['data'])):
        page_hash = hashlib.sha256(page_path.read_bytes()).hexdigest()
        assert page_hash == saved['settings']['pages'][page_idx]['sha256']
        sequence_pages[sequence_id].append((str(page_path.resolve()), page_hash))
        with Image.open(page_path) as image:
            width, height = image.size
        assert len(page['texts']) == len(page['ocr'])
        # Cached DETR boxes can extend slightly beyond the original page.
        boxes = check_boxes(page['texts'], width, height, allow_outside=True)
        gt_lines = labels[sequence_id]['pages'][page_idx]
        matches = [confident_match(text, gt_lines) for text in page['ocr']]
        gt_counts = Counter(match['gt_index'] for match in matches if match is not None)
        for text_idx, match in enumerate(matches):
            if match is None:
                excluded['weak_or_ambiguous_ocr'] += 1
                continue
            if gt_counts[match['gt_index']] != 1:
                excluded['reused_gt_item'] += 1
                continue
            gt = gt_lines[match['gt_index']]
            speaker = gt['speaker']
            if speaker == 'NARRATION':
                label = 1
            elif re.fullmatch(r'character[1-9][0-9]*', speaker):
                label = 0
            else:
                excluded['invalid_speaker'] += 1
                continue
            records.append(dict(sequence_id=sequence_id, page_idx=page_idx,
                page_path=str(page_path.resolve()), page_sha256=page_hash, text_idx=text_idx,
                bbox=boxes[text_idx].tolist(), ocr_text=page['ocr'][text_idx],
                gt_text=gt['text'], gt_speaker=speaker, label=label, **match))
assert records, 'No confident labels found.'
print('Accepted labels:', len(records), '| Excluded:', dict(excluded))


Accepted labels: 1490 | Excluded: {'weak_or_ambiguous_ocr': 665, 'invalid_speaker': 26, 'reused_gt_item': 2}


## Group shared pages before feature extraction
Keep overlapping sequences together. Assign the split after token matching.


In [4]:
# Group sequences sharing any page, including identical files under different paths.
parents = {seq: seq for seq in labels}


def find_group(seq):
    while parents[seq] != seq:
        parents[seq] = parents[parents[seq]]
        seq = parents[seq]
    return seq


page_owner = {}
for seq, pages in sequence_pages.items():
    for page_path, page_hash in pages:
        for key in (page_path, page_hash):
            if key in page_owner:
                parents[find_group(seq)] = find_group(page_owner[key])
            page_owner[key] = seq
groups = defaultdict(list)
for seq in labels:
    groups[find_group(seq)].append(seq)

print('Independent sequence groups:', len(groups))


Independent sequence groups: 80


## Load frozen Magi locally
OCR is disabled because the cache supplies it. Crop embeddings stay enabled to preserve the
checkpoint's existing association-head shapes, but the crop encoder is never run.

In [5]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
model, loading_info = AutoModel.from_pretrained(
    MAGI_PATH, trust_remote_code=True, local_files_only=True,
    disable_ocr=True, disable_detections=False, output_loading_info=True)
assert not loading_info['missing_keys'], loading_info['missing_keys']
assert not loading_info.get('mismatched_keys'), loading_info.get('mismatched_keys')
model = model.to(DEVICE).eval()
model.requires_grad_(False)
assert not any(p.requires_grad for p in model.parameters())
FEATURE_DIM = model.config.detection_model_config.d_model
model_files = {}
for name in ['config.json', 'configuration_magiv2.py', 'modelling_magiv2.py', 'processing_magiv2.py', 'utils.py', 'pytorch_model.bin']:
    path = Path(MAGI_PATH) / name
    hasher = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b''):
            hasher.update(chunk)
    model_files[name] = hasher.hexdigest()
model_identity = {'path': MAGI_PATH, 'sha256': model_files}
print('Feature dimension:', FEATURE_DIM, '| Trainable Magi parameters: 0')


c:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\torch\nn\modules\module.py:2593: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
c:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\torch\nn\modules\batchnorm.py:141: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  super()._load_from_state_dict(
c:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\torch\nn\modules\batchnorm.py:141: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to 

Feature dimension: 256 | Trainable Magi parameters: 0


In [6]:
@torch.no_grad()
def detect_text_tokens(page_path):
    with Image.open(page_path) as image:
        page = np.array(image.convert('L').convert('RGB'))
    height, width = page.shape[:2]
    inputs = model.processor.preprocess_inputs_for_detection([page])
    inputs = model.move_to_device(inputs)
    output = model._get_detection_transformer_output(**inputs)
    class_logits, normalized_boxes = model._get_predicted_bboxes_and_classes(output)
    tokens = model._get_predicted_obj_tokens(output)
    scores, classes = class_logits.max(-1)
    boxes = center_to_corners_format(normalized_boxes)
    boxes = boxes * boxes.new_tensor([width, height, width, height])
    query_ids = model.processor._get_indices_of_texts_to_keep(
        scores.sigmoid(), classes.long(), boxes, TEXT_DETECTION_THRESHOLD)[0]
    query_ids = torch.as_tensor(query_ids, dtype=torch.long, device=DEVICE)
    assert tokens.shape[1] == boxes.shape[1]
    text_boxes = boxes[0, query_ids].cpu().numpy()
    check_boxes(text_boxes, width, height, allow_outside=True)
    return text_boxes, tokens[0, query_ids].float().cpu(), query_ids.cpu().tolist()


def bbox_iou_matrix(first, second):
    first = np.asarray(first, dtype=np.float64).reshape(-1, 4)
    second = np.asarray(second, dtype=np.float64).reshape(-1, 4)
    top_left = np.maximum(first[:, None, :2], second[None, :, :2])
    bottom_right = np.minimum(first[:, None, 2:], second[None, :, 2:])
    intersection = np.maximum(bottom_right - top_left, 0).prod(axis=-1)
    first_area = (first[:, 2:] - first[:, :2]).prod(axis=-1)
    second_area = (second[:, 2:] - second[:, :2]).prod(axis=-1)
    union = first_area[:, None] + second_area[None, :] - intersection
    return intersection / np.maximum(union, 1e-12)


## Match boxes one-to-one and cache their exact query tokens

In [7]:
records_by_page = defaultdict(list)
for record in records:
    records_by_page[record['page_path']].append(record)
samples = []
best_ious = []
accepted_ious = []
failed = 0
page_detections = {}
for page_path, page_records in tqdm(records_by_page.items(), desc='Frozen DETR pages'):
    page_hash = page_records[0]['page_sha256']
    if page_hash not in page_detections:
        page_detections[page_hash] = detect_text_tokens(page_path)
    predicted_boxes, tokens, query_ids = page_detections[page_hash]
    if not len(predicted_boxes):
        best_ious.extend([0.0] * len(page_records))
        failed += len(page_records)
        continue
    ious = bbox_iou_matrix([r['bbox'] for r in page_records], predicted_boxes)
    best_ious.extend(ious.max(axis=1).tolist())
    eligible = ious >= MIN_BBOX_IOU
    rewards = np.where(eligible, len(page_records) + 1 + ious, -1.0)
    rewards = np.concatenate([rewards, np.zeros((len(page_records), len(page_records)))], axis=1)
    rows, columns = linear_sum_assignment(-rewards)
    matched = 0
    used_queries = set()
    for row, column in zip(rows, columns):
        if column >= len(predicted_boxes) or not eligible[row, column]:
            continue
        query_idx = query_ids[column]
        assert query_idx not in used_queries
        used_queries.add(query_idx)
        embedding = tokens[column].clone()
        assert embedding.shape == (FEATURE_DIM,) and torch.isfinite(embedding).all()
        sample = dict(page_records[row], query_idx=query_idx,
            detr_bbox=predicted_boxes[column].tolist(), bbox_iou=float(ious[row, column]),
            detr_embedding=embedding)
        samples.append(sample)
        accepted_ious.append(sample['bbox_iou'])
        matched += 1
    failed += len(page_records) - matched
assert samples, 'No boxes matched DETR tokens.'
print('Best IoU [min, 25%, median, 75%, max]:', np.quantile(best_ious, [0, .25, .5, .75, 1]).round(4))
print('Token matching | Successful:', len(samples), '| Failed:', failed)
print('Accepted IoU | Mean:', round(float(np.mean(accepted_ious)), 4), '| Min:', round(min(accepted_ious), 4))


Frozen DETR pages:   0%|          | 0/225 [00:00<?, ?it/s]

Best IoU [min, 25%, median, 75%, max]: [0.8007 0.9998 0.9999 1.     1.    ]
Token matching | Successful: 1490 | Failed: 0
Accepted IoU | Mean: 0.9954 | Min: 0.8007


## Audit and split surviving samples
Use surviving labels, never classifier scores, to choose the split.
Target 20% of narration samples first, then 20% of usable sequences.
Validation retains all accepted samples from its selected sequence groups.


In [8]:
# Print narration support at each stage and by narration-containing sequence.
gt_counts = Counter()
label_counts = Counter()
token_counts = Counter()
for seq, row in labels.items():
    gt_counts[seq] = sum(gt['speaker'] == 'NARRATION' for page in row['pages'] for gt in page)
for record in records:
    label_counts[record['sequence_id']] += record['label']
for sample in samples:
    token_counts[sample['sequence_id']] += sample['label']
print('Narration audit | GT:', sum(gt_counts.values()),
      '| Accepted labels:', sum(label_counts.values()), '| Matched tokens:', sum(token_counts.values()))
for seq in sorted(labels):
    if gt_counts[seq]:
        print(f'{seq} | GT {gt_counts[seq]} | Labels {label_counts[seq]} | Tokens {token_counts[seq]}')

usable_sequences = {sample['sequence_id'] for sample in samples}
group_ids = sorted(key for key, sequences in groups.items() if usable_sequences.intersection(sequences))
group_counts = defaultdict(Counter)
for sample in samples:
    group_counts[find_group(sample['sequence_id'])][sample['label']] += 1
total_counts = Counter(sample['label'] for sample in samples)
positive_groups = [key for key in group_ids if group_counts[key][1]]
assert len(positive_groups) >= 2, 'Need two independent narration groups.'
assert total_counts[0] > 1 and total_counts[1] > 1

# Search deterministic whole-group splits using label counts only.
rng = random.Random(SEED)
best_split = None
best_cost = None
for attempt in range(1000):
    order = rng.sample(group_ids, len(group_ids))
    candidate_sequences = set()
    candidate_counts = Counter()
    for key in order[:-1]:
        candidate_sequences.update(groups[key])
        candidate_counts.update(group_counts[key])
        if not all(0 < candidate_counts[label] < total_counts[label] for label in [0, 1]):
            continue
        narration_share = candidate_counts[1] / total_counts[1]
        sequence_share = len(candidate_sequences & usable_sequences) / len(usable_sequences)
        sample_share = sum(candidate_counts.values()) / len(samples)
        cost = (abs(narration_share - VALIDATION_FRACTION),
                abs(sequence_share - VALIDATION_FRACTION),
                abs(sample_share - VALIDATION_FRACTION))
        if best_cost is None or cost < best_cost:
            best_cost = cost
            best_split = set(candidate_sequences)
assert best_split is not None, 'Cannot split independent groups with both classes.'
validation_sequences = best_split
train_sequences = set(labels) - validation_sequences
assert train_sequences.isdisjoint(validation_sequences)
train_pages = {path for seq in train_sequences for path, _ in sequence_pages[seq]}
validation_pages = {path for seq in validation_sequences for path, _ in sequence_pages[seq]}
train_hashes = {value for seq in train_sequences for _, value in sequence_pages[seq]}
validation_hashes = {value for seq in validation_sequences for _, value in sequence_pages[seq]}
assert train_pages.isdisjoint(validation_pages)
assert train_hashes.isdisjoint(validation_hashes)
for sample in samples:
    sample['split'] = 'validation' if sample['sequence_id'] in validation_sequences else 'train'
validation_positive = sum(s['label'] for s in samples if s['split'] == 'validation')
validation_usable = len(validation_sequences & usable_sequences)
print(f'Validation narration: {validation_positive}/{total_counts[1]} '
      f'({validation_positive / total_counts[1]:.1%}) | Target: {VALIDATION_FRACTION:.0%}')
print(f'Validation usable sequences: {validation_usable}/{len(usable_sequences)} '
      f'({validation_usable / len(usable_sequences):.1%})')
print('Narration-containing sequences | Train:', sum(token_counts[seq] > 0 for seq in train_sequences),
      '| Validation:', sum(token_counts[seq] > 0 for seq in validation_sequences))


Narration audit | GT: 76 | Accepted labels: 63 | Matched tokens: 63
seq_032cbb4dc8686963 | GT 1 | Labels 0 | Tokens 0
seq_06ec0acde37dffa4 | GT 5 | Labels 3 | Tokens 3
seq_44b9a316b2919cfe | GT 1 | Labels 1 | Tokens 1
seq_72fdfff4b4353a42 | GT 4 | Labels 3 | Tokens 3
seq_91119fd54cd62066 | GT 1 | Labels 0 | Tokens 0
seq_a3bd36201a0cd61b | GT 20 | Labels 16 | Tokens 16
seq_ac5793f8168efdd1 | GT 9 | Labels 9 | Tokens 9
seq_b251c8e1ba8b91f8 | GT 7 | Labels 7 | Tokens 7
seq_d8bc3c6ac1bb6017 | GT 1 | Labels 0 | Tokens 0
seq_da71f7a842c91090 | GT 1 | Labels 0 | Tokens 0
seq_dcebdbbf614a9b4d | GT 3 | Labels 3 | Tokens 3
seq_ee880b9612205860 | GT 17 | Labels 15 | Tokens 15
seq_f164203a45862778 | GT 2 | Labels 2 | Tokens 2
seq_f1f5fb7a11b950f3 | GT 4 | Labels 4 | Tokens 4
Validation narration: 13/63 (20.6%) | Target: 20%
Validation usable sequences: 16/79 (20.3%)
Narration-containing sequences | Train: 6 | Validation: 4


## Rebalance training once and save

In [9]:
splits = {name: [i for i, sample in enumerate(samples) if sample['split'] == name]
          for name in ['train', 'validation']}
assert all({samples[i]['label'] for i in indices} == {0, 1} for indices in splits.values()), 'Token matching removed a class.'
assert set(splits['train']).isdisjoint(splits['validation'])
assert set(splits['train']) | set(splits['validation']) == set(range(len(samples)))
positives = [i for i in splits['train'] if samples[i]['label'] == 1]
negatives = [i for i in splits['train'] if samples[i]['label'] == 0]
rng = random.Random(SEED)
train_sample_indices = positives + rng.sample(negatives, min(len(negatives), int(NEGATIVE_RATIO * len(positives))))
rng.shuffle(train_sample_indices)
assert set(positives) <= set(train_sample_indices) <= set(splits['train'])
settings = dict(seed=SEED, negative_ratio=NEGATIVE_RATIO, min_bbox_iou=MIN_BBOX_IOU,
    magi_path=MAGI_PATH, match_threshold=MATCH_THRESHOLD, match_margin=MATCH_MARGIN,
    text_detection_threshold=TEXT_DETECTION_THRESHOLD, validation_fraction=VALIDATION_FRACTION,
    data_source='development', preprocessing='grayscale_then_RGB', matching='direct_unique',
    split_strategy='surviving_tokens_narration_and_sequence_v2', page_index_base=0)
data = dict(samples=samples, splits=splits, train_sample_indices=train_sample_indices,
    narration_audit={'gt': sum(gt_counts.values()), 'accepted_labels': sum(label_counts.values()),
                     'matched_tokens': sum(token_counts.values())},
    sequence_splits={'train': sorted(train_sequences), 'validation': sorted(validation_sequences)},
    settings=settings, feature_dim=FEATURE_DIM, model_identity=model_identity,
    token_matching={'successful': len(samples), 'failed': failed,
                    'mean_iou': float(np.mean(accepted_ious)), 'min_iou': min(accepted_ious)})
data['dataset_id'] = digest({'settings': settings, 'model': model_identity,
    'samples': [{k: v for k, v in sample.items() if k != 'detr_embedding'} for sample in samples],
    'splits': splits, 'train_sample_indices': train_sample_indices})
torch.save(data, ROOT / 'narration_data.pt')
print('Total accepted matched samples:', len(samples))
for title, indices in [('RAW TRAIN', splits['train']), ('USED TRAIN', train_sample_indices), ('VALIDATION', splits['validation'])]:
    counts = Counter(samples[i]['label'] for i in indices)
    print(f'{title} | Narration: {counts[1]} | Non-narration: {counts[0]}')
for name, sequences in data['sequence_splits'].items():
    narration_sequences = {samples[i]['sequence_id'] for i in splits[name] if samples[i]['label'] == 1}
    print(f'{name} sequences: {len(sequences)} | Narration-containing: {len(narration_sequences)}')
print('Saved:', ROOT / 'narration_data.pt')


Total accepted matched samples: 1490
RAW TRAIN | Narration: 50 | Non-narration: 1143
USED TRAIN | Narration: 50 | Non-narration: 100
VALIDATION | Narration: 13 | Non-narration: 284
train sequences: 64 | Narration-containing: 6
validation sequences: 16 | Narration-containing: 4
Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\isNarratorTextMagi\narration_data.pt
